# Positioned roughness elements, one STL per wind direction

Each case has its own transformed terrain, so the roughness array is regenerated and
re-draped per direction. `end_of_blocks` is the X where the roughness band stops for that
direction, applied as the bounding box end.

In [ ]:
import pathlib
import pprint

from cfdmod.io.geometry.STL import export_stl
from cfdmod.roughness import PositionParams, position_pattern

pp = pprint.PrettyPrinter()

cases = ["000", "090", "180", "270"]
end_of_blocks = [86, 82, 94, 94]
surfaces_names = ["terrain"]

cases_root_dir = pathlib.Path(
    "/mnt/disk01/prd-eng/cases/s1_consulting_nassu_test/025_BrookfieldAlceu/"
)

cfg_file = cases_root_dir / "simulation_data/configs_cfdmod/roughness_params_s24.yaml"
cfg = PositionParams.from_file(cfg_file)

output_path_root = cases_root_dir / "setup/STLs/STLs_files/"
results_path = cases_root_dir / "results/pilot/"
bodies_relative_path = pathlib.Path("000/bodies/lnas")

pp.pprint(cfg.model_dump())

In [ ]:
for case, end_blocks in zip(cases, end_of_blocks):
    surfaces = [
        (results_path / case) / (bodies_relative_path / f"{surface}.transformed.lnas")
        for surface in surfaces_names
    ]

    end = list(cfg.bounding_box.end)
    cfg.bounding_box.end = (end_blocks, end[1], end[2])

    triangles, normals = position_pattern(
        element_params=cfg.element_params,
        spacing_params=cfg.spacing_params,
        bounding_box=cfg.bounding_box,
        surfaces=surfaces,
    )

    print(f"{case}: {len(triangles) // 2} elements, box end {cfg.bounding_box.end}")
    export_stl((output_path_root / case) / "positioned_elements.stl", triangles, normals)